# E07 · 01 Query Scoping：案例 A 的候选定位与核查观察

p2 的 notebook 探索面：固定需求条件后，用 p1 产物定位种子候选，并跟踪逐项核查判断。

读入：

- `data/processed/e07/` 的 p0 语料与 p1 产物（只读）；
- `experiments/e07-p4a-case-pool/annotations/case-a.yml`（需求条件 + 判断记录，人工维护）。

**口径声明**：p1 的边与锚点是 v1 抽取线索；判断矩阵里的状态以 annotations 的人工核查为准。
本 notebook 不产出被引用的数字；成为文档论断前需下沉为脚本产物。

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yaml

import nbio

nbio.banner()

REPO = nbio.REPO_ROOT
P1 = nbio.OUT / "p1"
ANNOT = REPO / "experiments/e07-p4a-case-pool/annotations/case-a.yml"

spec = yaml.safe_load(ANNOT.read_text(encoding="utf-8"))
edges = [json.loads(l) for l in open(P1 / "resource_edges.jsonl")]
anchors = [json.loads(l) for l in open(P1 / "cite_anchors.jsonl")]
print(f"案例 {spec['case']}（spec {spec['spec_version']}）："
      f"{len(edges)} resource edges, {len(anchors)} cite anchors")

## 需求与满足条件

判断基准来自 annotations 的 query spec（版本化；不为排除某论文事后收紧）。
当前为草案，确认条件文本是 p2 的第一个决策点。

In [ ]:
print(spec["query"])
cond_df = pd.DataFrame(spec["conditions"])[["id", "label", "detail"]]
display(cond_df)
print("暂不纳入条件的维度：")
for n in spec["non_conditions"]:
    print(f"  - {n}")

## 视图 1 · 锚点反查

两类锚点：

- **资源边**：语料论文声明使用的 agent 环境/benchmark（WebArena、Mind2Web 等），
  使用这些环境的论文是"做 agent 任务"的候选集合；
- **引文锚点**：案例 A 的外部线索论文（AWM、Voyager 等）被语料哪些论文引用——
  引用者是"知道这条文献线"的候选。

两路的重叠与差集值得看：只用环境不引文献线的论文，多半是步骤 2 的困难候选来源。

In [ ]:
RESOURCE_ANCHORS = ["webarena", "visualwebarena", "workarena", "mind2web",
                    "osworld", "webshop", "alfworld", "assistantbench", "webcanvas"]
EXT_PAPERS = {  # 案例 A 外部线索 + 两个基础环境论文
    "2409.07429": "AWM", "2504.07079": "SkillWeaver", "2308.10144": "ExpeL",
    "2305.16291": "Voyager", "2306.07863": "Synapse",
    "2307.13854": "WebArena", "2306.06070": "Mind2Web",
}

res_hits = {}
for e in edges:
    for pat in RESOURCE_ANCHORS:
        if pat in e["norm_key"]:
            res_hits.setdefault(pat, set()).add(e["paper_id"])

cite_hits = {}
for a in anchors:
    for ax in a["arxiv_ids"]:
        ax = re.sub(r"v\d+$", "", ax.lower())
        if ax in EXT_PAPERS:
            cite_hits.setdefault(EXT_PAPERS[ax], set()).add(a["paper_id"])

rows = ([{"anchor": k, "type": "resource", "n_papers": len(v)} for k, v in res_hits.items()]
        + [{"anchor": k, "type": "citation", "n_papers": len(v)} for k, v in cite_hits.items()])
anchor_df = pd.DataFrame(rows).sort_values("n_papers", ascending=True)
fig = px.bar(anchor_df, x="n_papers", y="anchor", color="type", orientation="h",
             title="锚点 -> 语料内论文数（去重）")
fig.show()

anchor_papers = set().union(*res_hits.values(), *cite_hits.values())
print(f"锚点并集：{len(anchor_papers)} 篇语料论文")

## 视图 2 · 关键词扫面与候选漏斗

关键词扫面用 `paper_record.yml` 的标题 + 摘要（不读全文，保持轻量）。
词组分"轨迹/经验"与"复用/提炼"两组，同时命中两组才算主题接近。

漏斗口径：952 语料 → 锚点命中 → 关键词双组命中 → 两路并集（p2 候选工作集）。

In [ ]:
KW_TRAJ = ["trajector", "agent history", "past experience", "workflow", "skill"]
KW_REUSE = ["reus", "induc", "distill", "abstract", "retriev", "memory", "accumulat"]

kw_rows = []
for pdir in sorted(nbio.CORPUS.iterdir()):
    rec = yaml.safe_load((pdir / "layer4/paper_record.yml").read_text(encoding="utf-8"))
    meta = rec["paper_record"]["metadata"]
    text = (meta.get("title", "") + " " + (rec["paper_record"]
            .get("content_units", {}).get("abstract") or "")).lower()
    kw_rows.append({
        "paper_id": pdir.name,
        "title": meta.get("title", ""),
        "kw_traj": any(k in text for k in KW_TRAJ),
        "kw_reuse": any(k in text for k in KW_REUSE),
    })
kw_df = pd.DataFrame(kw_rows).set_index("paper_id")
kw_hit = set(kw_df.index[kw_df["kw_traj"] & kw_df["kw_reuse"]])
candidates = anchor_papers | kw_hit
print(f"关键词双组命中：{len(kw_hit)} 篇；候选工作集（并集）：{len(candidates)} 篇")

fig = go.Figure(go.Funnel(
    y=["952 语料", "锚点命中", "关键词双组命中", "候选工作集（并集）"],
    x=[len(kw_df), len(anchor_papers), len(kw_hit), len(candidates)],
    textinfo="value"))
fig.update_layout(title="p2 候选漏斗（结构定位，未核查）")
fig.show()

## 视图 3 · 判断矩阵（核查主视图）

候选 × 满足条件的状态矩阵，数据来自 annotations 的人工核查记录。
`unchecked` 之外的状态都应附证据（hover 可见）。

- 语料外线索（in_corpus=false）的核查需补外部材料；
- 语料内候选核查后追加进 annotations，重跑本 notebook 即更新矩阵。

In [ ]:
STATUS_COLOR = {"unchecked": 0, "insufficient_evidence": 1, "not_satisfied": 2,
                "partial": 3, "satisfied": 4}
conds = [c["id"] for c in spec["conditions"]]
cond_labels = [f"{c['id']} {c['label']}" for c in spec["conditions"]]

cand_rows, z, hover = [], [], []
for c in spec["candidates"]:
    name = f"{c['name']}（{'语料内' if c['in_corpus'] else '外部'}）"
    cand_rows.append(name)
    z.append([STATUS_COLOR[c["judgments"].get(cid, "unchecked")] for cid in conds])
    ev = "; ".join(e[:80] for e in c.get("evidence", [])) or "—"
    hover.append([f"{c['name']} / {cid}: {c['judgments'].get(cid, 'unchecked')}<br>{ev}"
                  for cid in conds])

fig = go.Figure(go.Heatmap(
    z=z, x=cond_labels, y=cand_rows, text=hover, hoverinfo="text",
    colorscale=[[0, "#eeeeee"], [0.25, "#f4d03f"], [0.5, "#e74c3c"],
                [0.75, "#3498db"], [1, "#2ecc71"]],
    colorbar=dict(tickvals=[0, 1, 2, 3, 4],
                  ticktext=list(STATUS_COLOR)), showscale=True))
fig.update_layout(title="案例 A 判断矩阵（annotations/case-a.yml）", height=500)
fig.show()

## 视图 4 · 关键词检索对照

"选择转折"论证需要记录：确认的候选能否被合理关键词检索直接命中。
下表对候选工作集给出 标题/摘要 层面的关键词命中情况；判断确认后（视图 3 的
种子正例），回看这里可回答"普通检索是否已足够"。

In [ ]:
view = kw_df.loc[sorted(candidates)].copy()
view["via_anchor"] = view.index.isin(anchor_papers)
view = view.sort_values(["via_anchor", "kw_traj", "kw_reuse"], ascending=False)
print(f"候选工作集 {len(view)} 篇："
      f"锚点 {view['via_anchor'].sum()}，关键词双组 {((view['kw_traj']) & (view['kw_reuse'])).sum()}，"
      f"两组皆无（潜在'检索难命中'）{(~(view['kw_traj']) & ~(view['kw_reuse'])).sum()}")
display(view.head(40))

## 用法与边界

- 核查流程：从视图 1/2 的候选出发回原始材料（mineru 全文、resource_records、
  cite contexts），判断写入 `annotations/case-a.yml`，重跑本 notebook。
- 本 notebook 的所有数字都是探索性统计；进入文档引用前需下沉为 `src/e07/` 脚本产物。
- 关键词组与锚点清单是分析假设，改动请记录原因（避免事后收紧等价于改条件）。